In [ ]:
!pip install modal

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["MODAL_TOKEN_ID"] = userdata.get('token-id')
    os.environ["MODAL_TOKEN_SECRET"] = userdata.get('token-secret')
    print("✅ Authentifizierung für Modal geladen!")
except Exception as e:
    print(f"❌ Fehler: {e}. Hast du das Schlüssel-Icon links konfiguriert?")

tid = os.environ.get("MODAL_TOKEN_ID")
tsec = os.environ.get("MODAL_TOKEN_SECRET")
print(f"Token ID startet mit: {tid[:4]}... (Länge: {len(tid)})")
print(f"Token Secret startet mit: {tsec[:4]}... (Länge: {len(tsec)})")



✅ Authentifizierung für Modal geladen!
Token ID startet mit: ak-b... (Länge: 25)
Token Secret startet mit: as-p... (Länge: 25)


In [ ]:
import os
from google.colab import userdata

# 1. Token aus den Colab-Secrets laden
hf_token_value = userdata.get('HF_TOKEN')

# 2. Das Modal-Secret erstellen, ohne den Token im Code zu zeigen
# Wir nutzen die f-String Syntax für den System-Befehl
if hf_token_value:
    !modal secret create vbot_modal_huggingface HF_TOKEN='{hf_token_value}' --force
    print("✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!")
else:
    print("❌ Fehler: HF_TOKEN wurde nicht in den Colab-Secrets gefunden.")

Created a new secret 'vbot_modal_huggingface' with the key 'HF_TOKEN'

Use it in your Modal app:

                                                                                
@app.function(secrets=[modal.Secret.from_name("vbot_modal_huggingface")])       
def some_function():                                                            
    os.getenv("HF_TOKEN")                                                       
                                                                                
✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!


In [ ]:

!modal profile list

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━┓
┃   ┃ Profile ┃ Workspace ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━┩
└───┴─────────┴───────────┘
Using matthias-nollek workspace based on environment variables


In [ ]:
%%writefile app.py
import modal
import os
import uuid
import asyncio
import json
from fastapi import FastAPI, WebSocket
from starlette.responses import HTMLResponse

# 1. Konfiguration
MAX_TOKENS = 8192
MAX_NEW_TOKENS = 2048
#MODEL_ID = "neuralmagic/Mistral-Nemo-12B-Instruct-v1-quantized.w4a16"
MODEL_ID = "mistralai/Mistral-Nemo-Instruct-FP8-2407"
SYSTEM_PROMPT = "You are a helpful assistant..."

# 2. Image Definition
vllm_image = (
    modal.Image.debian_slim(python_version="3.10")
    .pip_install(
        "vllm>=0.6.0",
        "transformers>=4.44.0",
        "tokenizers>=0.19.0",
        "fastapi",
        "starlette",
        "hf_transfer"
    )
    .env({
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "VLLM_LOGGING_LEVEL": "ERROR",
        "VLLM_USE_V1": "0"
    })
)

app = modal.App("twilio-voice-nemo")

# 3. Der Haupt-Service
@app.cls(
    image=vllm_image,
    gpu="A10G",
    quantization="awq", # WICHTIG
    secrets=[modal.Secret.from_name("vbot_modal_huggingface")],
    scaledown_window=10
)
class TwilioChatBot:
    @modal.enter()
    def load_engine(self):
        from vllm import AsyncEngineArgs, AsyncLLMEngine
        from transformers import AutoTokenizer

        engine_args = AsyncEngineArgs(
            model=MODEL_ID,
            gpu_memory_utilization=0.90,
            max_model_len=MAX_TOKENS,
            trust_remote_code=True,
            enforce_eager=True
        )
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
        # WICHTIG: Tokenizer der Klasse zuweisen
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    @modal.asgi_app()
    def fastapi_app(self):
        web_app = FastAPI()

        @web_app.post("/start_call")
        async def start_call():
            # Dynamische URL Auflösung
            service_url = f"{modal.current_app().app_id}--{modal.current_app().name}-fastapi_app.modal.run"
            tmpl = f"""<?xml version="1.0" encoding="UTF-8"?>
<Response>
  <Connect>
    <ConversationRelay url="wss://{service_url}/ws" welcomeGreeting="Hi! I'm Jane. Just chat with me!!"></ConversationRelay>
  </Connect>
</Response>"""
            return HTMLResponse(content=tmpl, media_type="application/xml")

        @web_app.websocket("/ws")
        async def websocket_endpoint(websocket: WebSocket):
            await websocket.accept()
            queue = asyncio.Queue()
            # Initialer Verlauf mit System Prompt
            history = [{"role": "system", "content": SYSTEM_PROMPT}]

            async def llm_request(message):
                from vllm import SamplingParams
                sampling_param = SamplingParams(max_tokens=MAX_NEW_TOKENS)

                prompt = self.tokenizer.apply_chat_template(
                    history + [dict(role="user", content=message)],
                    tokenize=False, add_generation_prompt=True,
                )
                request_id = uuid.uuid4().hex
                replies = []

                try:
                    stream = await self.engine.add_request(request_id, prompt, sampling_param)
                    cursor = 0
                    async for request_output in stream:
                        text = request_output.outputs[0].text
                        out = text[cursor:]
                        replies.append(out)
                        await websocket.send_json({"type": "text", "token": out, "last": False})
                        cursor = len(text)
                except asyncio.CancelledError:
                    await self.engine.abort(request_id)
                    raise
                finally:
                    reply = "".join(replies)
                    history.append(dict(role="user", content=message))
                    history.append(dict(role="assistant", content=reply))
                    await websocket.send_json({"type": "text", "token": "", "last": True})

            async def read_from_socket():
                async for data in websocket.iter_json():
                    await queue.put(data)

            async def process_logic():
                input_buffer = []
                llm_task = None
                while True:
                    data = await queue.get()
                    if data["type"] == "prompt":
                        input_buffer.append(data["voicePrompt"])
                        if data.get("last"):
                            message = " ".join(input_buffer)
                            input_buffer = []
                            if llm_task: llm_task.cancel()
                            llm_task = asyncio.create_task(llm_request(message))
                    elif data["type"] == "interrupt":
                        input_buffer = []
                        if llm_task: llm_task.cancel()

            await asyncio.gather(read_from_socket(), process_logic())

        return web_app

Overwriting app.py


In [ ]:
#!python3 -m py_compile app.py

In [ ]:
!modal deploy app.py

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/app.py:36 in <module>                                               │
│                                                                              │
│    35 # 3. Der Haupt-Service                                                 │
│ ❱  36 @app.cls(                                                              │
│    37 │   image=vllm_image,                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
TypeError: _App.cls() got an unexpected keyword argument 'quantization'
